# 2 · Inside the agglomerate model

In the agglomerate model each cathode node holds porous spheres (agglomerates, radius 5 µm) made of small crystals (radius 200 nm). The electrolyte fills the pores between agglomerates and the pores inside them. The two scales are solved together (docs/model.md section 6):

- the **electrode** carries salt and current between the foil and the current collector;
- each **agglomerate** takes electrolyte, current and electrons in through its surface, and its crystals react with the electrolyte in its pores.

This notebook looks at the state at the end of a 2C discharge, with the default (Bruggeman) transport inside the agglomerates and with a much slower one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nmc_model.agglomerate import AggParams
from nmc_model.agglomerate_corrected import CorrectedModel, run_corrected

p = AggParams(mode="corrected", C_rate=2.0, nj=30, nja=12, dt_s=5.0)
slow = AggParams(mode="corrected", C_rate=2.0, nj=30, nja=12, dt_s=5.0, tau_agg=2240.0)   # 1000x Bruggeman
r, r_slow = run_corrected(p), run_corrected(slow)
m = CorrectedModel(p)
st, st_slow = r.final_state, r_slow.final_state
for name, rr in (("default", r), ("slow agglomerate transport", r_slow)):
    print(name, ":", rr.exit_reason, "after", round(rr.array[-1, 0] * 60, 1), "min")

## Across the electrode

The electrolyte concentration in the macro-pores, from the foil (left) through the separator to the current collector (right).

In [ ]:
x = m.mesh.x * 1e4
plt.plot(x, m.conc(st)[0] * 1e3)
plt.axvline(p.L_sep * 1e4, color="gray", lw=0.5)
plt.xlabel("x [µm]"); plt.ylabel("c [mol/L]"); plt.title("electrolyte at the end of a 2C discharge")
plt.show()

## Inside the agglomerates

The crystal lithiation x = c_s/(ρ/M) along the radius of the agglomerates at the front (next to the separator), the middle and the back (next to the current collector). The center is r = 0, the surface r = R.

With the default transport the salt reaches the agglomerate centers easily, and every crystal lithiates almost equally. With 1000 times slower transport in the agglomerate pores, the salt inside is depleted: the crystals near the surface fill first, and the discharge ends early.

In [ ]:
r_um = m.xa * 1e4
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for a_, s_, title in ((ax[0], st, "default"), (ax[1], st_slow, "slow agglomerate transport")):
    for k, label in ((0, "front"), (m.nl // 2, "middle"), (m.nl - 1, "back")):
        a_.plot(r_um, m.cs(s_)[k] / p.mol_vol, label=label)
    a_.set_xlabel("r [µm]"); a_.set_title(title)
ax[0].set_ylabel("x in LixNMC"); ax[0].legend(); plt.show()

And the electrolyte inside the same agglomerates: the pores inside an agglomerate are fed only through its surface.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for a_, s_, title in ((ax[0], st, "default"), (ax[1], st_slow, "slow agglomerate transport")):
    for k, label in ((0, "front"), (m.nl // 2, "middle"), (m.nl - 1, "back")):
        a_.plot(r_um, m.conc(s_)[1][k] * 1e3, label=label)
    a_.set_xlabel("r [µm]"); a_.set_title(title)
ax[0].set_ylabel("c in the agglomerate pores [mol/L]"); ax[0].legend(); plt.show()

## Conservation

The coupling conserves lithium and salt exactly: the lithium in the crystals equals the charge passed, and the salt in both pore scales is constant (tests/test_agglomerate_corrected.py).

In [ ]:
v_one = 4 / 3 * np.pi * p.R_agg ** 3
li = ((((m.cs(st) - p.cs_init) * m.dV).sum(axis=1) * (1 - p.eps_agg) / v_one * m.v_agg) * m.mesh.dx[m.nodes]).sum()
passed = r.array[-1, 0] * 3600 * 2.0 * p.i_1C / p.F
print("lithium in crystals / charge passed =", li / passed)